## 3. Preparação para Tokenização (Pré-processamento)

* **Input**: CSV contendo todos os enunciados, alternativas e gabaritos

* **Output**: CSV com as colunas pré-processadas -> *numero_questao*, *enunciado*, *alternativas*,*nu_param_B*, *gabarito*, *ano*, *enunciado_limpo* e *alternativas_limpo*

In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

In [6]:
# Download stopwords (palavras de parada)
nltk.download('stopwords')
stop_words = set(stopwords.words('portuguese'))

# Dicionário de fórmulas químicas
formula_dict = {
    "CO2": "fórmula química do gás carbônico",
    "H2O": "fórmula química da água",
    "NaCl": "fórmula química do sal de cozinha",
    "O2": "fórmula química do gás oxigênio",
    "H2": "fórmula química do gás hidrogênio",
    "CH4": "fórmula química do metano",
    "NH3": "fórmula química da amônia",
    "C6H12O6": "fórmula química da glicose"
}

# Função para substituir fórmulas químicas
def substituir_formulas(text):
    if not isinstance(text, str):
        return ""
    
    for formula, descricao in formula_dict.items():
        # usar regex com bordas para evitar casos como CH4D ou CO2a
        text = re.sub(rf'\b{re.escape(formula)}\b', descricao, text)
    return text

# Função de limpeza do texto com substituição de fórmulas
def clean_text(text):
    """Processando texto de acordo com o Protocolo Primi (2021), citado no artigo"""
    if not isinstance(text, str):
        return ""

    text = substituir_formulas(text)  # Substituir fórmulas antes do restante da limpeza
    words = re.findall(r'\b[a-zA-Zà-úÀ-ÚüÜ]+\b', text.lower())
    words = [word for word in words if word not in stop_words]
    return " ".join(sorted(set(words)))

# Função de limpeza das alternativas com substituição de fórmulas
def clean_alternatives(alt_text):
    """Processa as alternativas mantendo as letras (A:, B:, etc.) e limpando o conteúdo."""
    if not isinstance(alt_text, str):
        return ""
    
    alt_text = substituir_formulas(alt_text)
    alternatives = re.split(r'(?=[A-E]: )', alt_text)
    cleaned_alts = []
    
    for alt in alternatives:
        if ": " in alt:
            key, value = alt.split(": ", 1)
            cleaned_value = clean_text(value)
            cleaned_alts.append(f"{key}: {cleaned_value}")
    
    return "; ".join(cleaned_alts)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ciziks/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df = pd.read_csv("../data/processed/full_enem_data.csv")

# Aplicando pré-processamento no enunciado e alternativas
df["enunciado_limpo"] = df["enunciado"].apply(clean_text)
df["alternativas_limpo"] = df["alternativas"].apply(clean_alternatives)

# Save cleaned CSV
df.to_csv("../data/processed/clean_enem_data.csv", index=False)
print("Processing complete. File saved as 'cleaned_data.csv'.")


Processing complete. File saved as 'cleaned_data.csv'.


## Removendo Fórmulas Químicas

In [ ]:
import pandas as pd
import re

# Carregar o arquivo CSV
file_path = "/mnt/data/enem_2010.csv"
df = pd.read_csv(file_path, header=None, names=["ID", "Texto"])

# Dicionário de fórmulas químicas para substituir por extenso
formula_dict = {
    "CO2": "fórmula química do gás carbônico",
    "H2O": "fórmula química da água",
    "NaCl": "fórmula química do sal de cozinha",
    "O2": "fórmula química do gás oxigênio",
    "H2": "fórmula química do gás hidrogênio",
    "CH4": "fórmula química do metano",
    "NH3": "fórmula química da amônia",
    "C6H12O6": "fórmula química da glicose"
}

# Função para substituir fórmulas químicas por suas descrições
def substituir_formulas(texto):
    for formula, descricao in formula_dict.items():
        texto = re.sub(rf'\b{formula}\b', descricao, texto)
    return texto

# Aplicar a substituição
df["Texto"] = df["Texto"].apply(substituir_formulas)

# Salvar o DataFrame editado
output_path = "/mnt/data/enem_2010_editado.csv"
df.to_csv(output_path, index=False)

output_path
